<a href="https://colab.research.google.com/github/shefiramarizcha62-sudo/flyrank-ml-portfolio/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shefiramarizcha62-sudo/flyrank-ml-portfolio/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("FlyRank-ML")

print("HF token loaded:", HF_TOKEN is not None)

con = duckdb.connect()

print("DuckDB connected:", con is not None)

con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

print("Hugging Face authentication configured.")

HF token loaded: True
DuckDB connected: True
Hugging Face authentication configured.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I use a Random Forest model for this week's modeling experiment.

The baseline from ML-07 is a rule-based ranking score built from search visibility,
CTR opportunity, and search position. The purpose of the ML model is not to reproduce
the baseline rule, but to test whether a nonlinear model can identify a more useful
ranking pattern from the same decision-moment signals.

Random Forest is selected because it can model nonlinear relationships between
search impressions, clicks/CTR, and search position without requiring a linear
relationship between the features and the outcome. It also provides feature
importance that can be used to interpret which signals contribute most to the model.

The model will be compared against the ML-07 baseline on the same evaluation data
and using the same evaluation objective. Future-window information, label-derived
features, product flags, and the baseline score itself will not be used as model
features.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*



I will use a time-based split to separate information available at the decision moment from the future outcome.

- **Feature window:** March 2026
- **Outcome window:** April 2026
- **Decision moment:** the end of March 2026
- **Training data:** March features with an April outcome
- **Validation/test:** a time-separated evaluation rather than a random split

This design prevents future April information from being used as a feature when making the decision at the end of March.

In [3]:
# ==================================================
# SECTION 2 — SPLIT DESIGN
# ==================================================

import numpy as np
import pandas as pd


# ==================================================
# 1. Load March feature window
# ==================================================
# March is the information available at the
# decision moment.

feature_frame = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_engaged_sessions

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )

    WHERE month = '2026-03'
""").df()


print("March feature rows:", len(feature_frame))
print(
    "March date range:",
    feature_frame["report_date"].min(),
    "to",
    feature_frame["report_date"].max()
)


# ==================================================
# 2. Aggregate March features to client-content level
# ==================================================
# This follows the same aggregation logic used
# when building the ML-07 baseline.

march_features = (
    feature_frame
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean"),
        ga4_pageviews=("ga4_pageviews", "sum"),
        ga4_engaged_sessions=("ga4_engaged_sessions", "sum")
    )
)


# March CTR is calculated only from March data.
march_features["march_ctr"] = (
    march_features["gsc_clicks"]
    / march_features["gsc_impressions"].replace(0, np.nan)
).fillna(0)


print("\nMarch client-content pairs:", len(march_features))


# ==================================================
# 3. Load April outcome window
# ==================================================
# April is NOT used as a model feature.
# It is used only to define the future outcome.

april_outcome = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet'
    )

    WHERE month = '2026-04'
""").df()


print("\nApril outcome rows:", len(april_outcome))
print(
    "April date range:",
    april_outcome["report_date"].min(),
    "to",
    april_outcome["report_date"].max()
)


# ==================================================
# 4. Aggregate April outcome
# ==================================================

april_outcome = (
    april_outcome
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        april_impressions=("gsc_impressions", "sum"),
        april_clicks=("gsc_clicks", "sum")
    )
)


# ==================================================
# 5. Calculate future April CTR
# ==================================================

april_outcome["april_ctr"] = (
    april_outcome["april_clicks"]
    / april_outcome["april_impressions"].replace(0, np.nan)
).fillna(0)


# ==================================================
# 6. Join March features with April outcome
# ==================================================
# Inner join keeps only content pairs that have
# both March features and an April outcome.

model_df = march_features.merge(
    april_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)


print("\nModeling rows:", len(model_df))
print(
    "Unique client-content pairs:",
    model_df[
        ["client_hash_id", "content_hash_id"]
    ].drop_duplicates().shape[0]
)


# ==================================================
# 7. Define future opportunity label
# ==================================================
# The label represents a future CTR opportunity:
# meaningful April visibility (>100 impressions)
# combined with low April CTR (<0.1%).
#
# These thresholds are consistent with the
# signal audit and ML-07 baseline.
#
# IMPORTANT:
# April information is used ONLY for the label,
# never as a model feature.

model_df["future_opportunity"] = (
    (model_df["april_impressions"] > 100)
    & (model_df["april_ctr"] < 0.001)
).astype(int)


# ==================================================
# 8. Inspect outcome distribution
# ==================================================

print("\nFuture opportunity distribution:")
print(
    model_df["future_opportunity"]
    .value_counts()
    .sort_index()
)

print("\nFuture opportunity percentage:")
print(
    model_df["future_opportunity"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)


# ==================================================
# 9. Preview
# ==================================================

model_df[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "march_ctr",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_engaged_sessions",
        "april_impressions",
        "april_clicks",
        "april_ctr",
        "future_opportunity"
    ]
].head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March feature rows: 9841378
March date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00

March client-content pairs: 331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


April outcome rows: 10424730
April date range: 2026-04-01 00:00:00 to 2026-04-30 00:00:00

Modeling rows: 331436
Unique client-content pairs: 331436

Future opportunity distribution:
future_opportunity
0    280002
1     51434
Name: count, dtype: int64

Future opportunity percentage:
future_opportunity
0    84.48
1    15.52
Name: proportion, dtype: float64


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,march_ctr,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions,april_impressions,april_clicks,april_ctr,future_opportunity
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,0.000000,NaN,0,0,0,0,0.000000,0
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,0.000000,NaN,0,0,0,0,0.000000,0
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,0.000000,NaN,0,0,0,0,0.000000,0
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,0.000000,9.000000,0,0,15,0,0.000000,0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,0.000000,NaN,0,0,0,0,0.000000,0
5,client_0797ff3a1fc9a6a5,content_0317b24cc1ff5c5d,0,0,0.000000,NaN,0,0,0,0,0.000000,0
6,client_0797ff3a1fc9a6a5,content_044c54ec4adcc4b2,0,0,0.000000,NaN,0,0,0,0,0.000000,0
7,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331,2,0.006042,14.129210,0,0,561,3,0.005348,0
8,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,0,0.000000,9.225529,0,0,0,0,0.000000,0
9,client_0797ff3a1fc9a6a5,content_07573a1cc2034981,0,0,0.000000,NaN,0,0,0,0,0.000000,0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I train a Random Forest classifier using only March decision-moment features and the April future-opportunity label.

The model is compared with the ML-07 rule-based baseline on the same client-content pairs and the same future outcome. The baseline score is treated as a ranking score, while the Random Forest predicted probability is used as the model ranking score.

The baseline score, reason codes, action labels, and April outcome fields are excluded from the model features to avoid leakage.

### 1. Define model features


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "march_ctr",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

X = model_df[feature_columns].copy()
y = model_df["future_opportunity"].copy()

print("Feature columns:")
print(feature_columns)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Feature columns:
['gsc_impressions', 'gsc_clicks', 'march_ctr', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']

X shape: (331436, 6)
y shape: (331436,)

Target distribution:
future_opportunity
0    280002
1     51434
Name: count, dtype: int64


### 2. Grouped validation by client

In [7]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer

# --------------------------------------------------
# 1. Handle missing feature values
# --------------------------------------------------

imputer = SimpleImputer(strategy="median")

X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns,
    index=X.index
)

# --------------------------------------------------
# 2. Group by client
# --------------------------------------------------

groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X_imputed,
        y,
        groups=groups
    )
)

X_train = X_imputed.iloc[train_idx]
X_test = X_imputed.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]


print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print(
    "Train clients:",
    groups_train.nunique()
)

print(
    "Test clients:",
    groups_test.nunique()
)

print(
    "\nClient overlap:",
    len(
        set(groups_train)
        .intersection(set(groups_test))
    )
)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(4))

Train rows: 300879
Test rows: 30557
Train clients: 44
Test clients: 11

Client overlap: 0

Train target distribution:
future_opportunity
0    0.8467
1    0.1533
Name: proportion, dtype: float64

Test target distribution:
future_opportunity
0    0.8262
1    0.1738
Name: proportion, dtype: float64


### 3. Train Random Forest

In [8]:
# ==================================================
# 3B. Train Random Forest
# ==================================================

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

print("Random Forest training completed.")
print("Number of trees:", rf_model.n_estimators)
print("Max depth:", rf_model.max_depth)

Random Forest training completed.
Number of trees: 200
Max depth: 12


### 4. Generate model score

In [9]:
# ==================================================
# 3C. Generate model score
# ==================================================

# Probability of future opportunity
model_score = rf_model.predict_proba(X_test)[:, 1]

test_results = model_df.iloc[test_idx][
    [
        "client_hash_id",
        "content_hash_id",
        "future_opportunity"
    ]
].copy()

test_results["model_score"] = model_score


# ==================================================
# 3D. Recreate ML-07 baseline score
# ==================================================
# IMPORTANT:
# The baseline must use only March decision-moment signals.
# No April information is used here.

baseline_test = model_df.iloc[test_idx].copy()

baseline_test["visibility_score"] = (
    baseline_test["gsc_impressions"].rank(pct=True)
)

baseline_test["ctr_opportunity"] = (
    1 - baseline_test["march_ctr"].rank(pct=True)
)

position_rank = baseline_test[
    "gsc_avg_position"
].where(
    baseline_test["gsc_avg_position"] > 0
)

baseline_test["position_opportunity"] = (
    1 - position_rank.rank(pct=True)
).fillna(0)

baseline_test["baseline_score"] = (
    baseline_test["visibility_score"]
    + baseline_test["ctr_opportunity"]
    + baseline_test["position_opportunity"]
) / 3

test_results["baseline_score"] = (
    baseline_test["baseline_score"].values
)


# ==================================================
# 3E. Compare ranking performance
# ==================================================

model_auc = roc_auc_score(
    test_results["future_opportunity"],
    test_results["model_score"]
)

baseline_auc = roc_auc_score(
    test_results["future_opportunity"],
    test_results["baseline_score"]
)

model_ap = average_precision_score(
    test_results["future_opportunity"],
    test_results["model_score"]
)

baseline_ap = average_precision_score(
    test_results["future_opportunity"],
    test_results["baseline_score"]
)


comparison = pd.DataFrame({
    "method": [
        "ML-07 Baseline",
        "Random Forest"
    ],
    "ROC_AUC": [
        baseline_auc,
        model_auc
    ],
    "Average_Precision": [
        baseline_ap,
        model_ap
    ]
})

comparison

,method,ROC_AUC,Average_Precision
0,ML-07 Baseline,0.738707,0.318966
1,Random Forest,0.921380,0.679493


### Model vs baseline result

The Random Forest substantially outperformed the ML-07 rule-based baseline on the same client-held-out test set.

The baseline achieved a ROC-AUC of 0.7387 and an Average Precision of 0.3190, while Random Forest achieved 0.9214 ROC-AUC and 0.6795 Average Precision.

This indicates that the nonlinear model provides substantially stronger ranking performance for identifying content associated with the April future-opportunity label. The result is directional rather than causal: it shows that the model ranks the observed future outcome better than the baseline under this validation design.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [10]:
# ==================================================
# SECTION 4 — ERRORS AND INTERPRETATION
# ==================================================

# --------------------------------------------------
# 1. Generate test predictions
# --------------------------------------------------

test_results["predicted_probability"] = test_results["model_score"]

test_results["predicted_class"] = (
    test_results["predicted_probability"] >= 0.50
).astype(int)


# --------------------------------------------------
# 2. Identify errors
# --------------------------------------------------

false_positives = test_results[
    (test_results["future_opportunity"] == 0)
    & (test_results["predicted_class"] == 1)
].copy()

false_negatives = test_results[
    (test_results["future_opportunity"] == 1)
    & (test_results["predicted_class"] == 0)
].copy()


print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))


# --------------------------------------------------
# 3. Feature importance
# --------------------------------------------------

feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)


print("\nFeature importance:")
print(feature_importance)


# --------------------------------------------------
# 4. Show strongest false positives
# --------------------------------------------------

print("\nTop false positives:")

print(
    false_positives[
        [
            "client_hash_id",
            "content_hash_id",
            "future_opportunity",
            "model_score"
        ]
    ]
    .sort_values(
        "model_score",
        ascending=False
    )
    .head(10)
)


# --------------------------------------------------
# 5. Show strongest false negatives
# --------------------------------------------------

print("\nTop false negatives:")

print(
    false_negatives[
        [
            "client_hash_id",
            "content_hash_id",
            "future_opportunity",
            "model_score"
        ]
    ]
    .sort_values(
        "model_score",
        ascending=True
    )
    .head(10)
)

False positives: 4762
False negatives: 361

Feature importance:
                feature  importance
0       gsc_impressions    0.618456
1      gsc_avg_position    0.179069
2             march_ctr    0.102659
3            gsc_clicks    0.070816
4         ga4_pageviews    0.026693
5  ga4_engaged_sessions    0.002308

Top false positives:
                 client_hash_id           content_hash_id  future_opportunity  \
303523  client_e547b89c05043229  content_b5632e85c2180dea                   0   
305083  client_e547b89c05043229  content_e2d47b2609d4b274                   0   
302098  client_e547b89c05043229  content_8ebf54dc6ab0210b                   0   
301609  client_e547b89c05043229  content_80f1619d2f4038da                   0   
302982  client_e547b89c05043229  content_a5c36b795156a9e9                   0   
298192  client_e547b89c05043229  content_2430ccc68e7e7b24                   0   
305503  client_e547b89c05043229  content_ee5b5620085c8e99                   0   
304121  client

### Error analysis and interpretation

The Random Forest produced 4,762 false positives and 361 false negatives on the client-held-out test set. The larger number of false positives indicates that the model sometimes prioritizes content that appears promising from the March signals but does not become a future opportunity in April. In contrast, the relatively small number of false negatives suggests that the model misses fewer positive opportunities.

Feature importance shows that the model relies most strongly on `gsc_impressions` (61.85%), followed by `gsc_avg_position` (17.91%) and `march_ctr` (10.27%). This indicates that search visibility is the dominant signal in the model, while search position and CTR provide additional information. GA4 features contribute much less to the model, particularly `ga4_engaged_sessions` (0.23%).

The strongest false positives had high predicted probabilities despite having a negative future label. These cases represent content that looked promising based on March search signals but did not meet the April opportunity definition. The false negatives represent a smaller group of future opportunities that the model failed to prioritize, suggesting that some opportunities are not well captured by the available March search and engagement signals.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.